[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr.ipynb)

# Light Past a Rotating Black Hole in Gauge Theory Gravity

This example traces light rays and polarized emission past a rotating Kerr black hole in gauge theory gravity. Light rays pass the hole across a sweep of black-hole spins, and a camera views a glowing equatorial accretion disk through a rotating linear polarizer as orbiting clumps wind into trailing spirals. Gravity bends the rays, captures those inside the photon sphere, and drags the rest with the hole's rotation, lensing the far side of the disk around the central shadow and shifting the approaching gas toward higher frequency and brightness.

Events and directions are modelled as intrinsic antivectors, while momenta and spatial gradients are modelled as extrinsic vectors that pair with directions via `momentum & direction`, and the hole's spin per unit mass is modelled as an intrinsic spatial patch whose dual inverse gives the turning plane. Following [Doran (2000)](https://arxiv.org/abs/gr-qc/9910099), [Francis and Kosowsky (2004)](https://arxiv.org/pdf/gr-qc/0311007), and the Cartesian Kerr–Schild profile of [Pelle et al. (2022)](https://academic.oup.com/mnras/article/515/1/1316/6631564), a position gauge field `Antivector + profile * null * (null_plane & Antivector)` maps flat local-frame directions to Kerr directions; because the null dyad squares to zero, subtracting it gives the exact inverse, and minus half the flat square of `gauge(profile, null)(-momentum.dual())` gives the ray Hamiltonian. Pulling the momentum back through `null_gradient.adjugate()(momentum)` combines with the profile gradient to drive the momentum rate without coordinate components, a rotation gauge bivector transports the camera screen along each ray via `-connection.commutator(screen)`, and pairing unit analyzer slices across the disk's polarization covariance `analyzers & coherency(analyzers)` reads transmitted power. Companion notebooks develop [the metric and Levi-Civita formulation](kerr_metric.ipynb), [a side-by-side comparison of the two formulations](kerr_comparison.ipynb), and [interactive WebGPU ray tracing](kerr_gpu.ipynb).

In tensor-index notation, the Cartesian Kerr–Schild metric $g_{\mu\nu} = \eta_{\mu\nu} - 2f\,l_\mu l_\nu$ and its inverse $g^{\mu\nu} = \eta^{\mu\nu} + 2f\,l^\mu l^\nu$ enter the ray Hamiltonian $H = \tfrac{1}{2} g^{\mu\nu} p_\mu p_\nu$, whose coordinate equations $\dot{x}^\mu = g^{\mu\nu} p_\nu$ and $\dot{p}_\mu = -\tfrac{1}{2} (\partial_\mu g^{\alpha\beta}) p_\alpha p_\beta$ expand metric derivatives component by component while parallel transport integrates $\dot{s}^\mu_a = -\Gamma^\mu{}_{\alpha\beta} \dot{x}^\alpha s^\beta_a$ through the forty independent components of the Christoffel symbols $\Gamma^\mu{}_{\alpha\beta} = \tfrac{1}{2} g^{\mu\nu}(\partial_\alpha g_{\nu\beta} + \partial_\beta g_{\nu\alpha} - \partial_\nu g_{\alpha\beta})$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack, where
from numga.algebras import STA as ga
from examples.util.animation import save_figure, show_animation, show_states
from examples.relativity.kerr import render
from examples.relativity.kerr.plumbing import resolve, trace

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
# Events and directions are intrinsic antivectors; momenta, gradients and coordinate slices are extrinsic vectors.
Vector = ga.gatype.vector()                                  # Vector+
Bivector = ga.gatype.bivector()                              # Bivector+
Antivector = ga.gatype.antivector()                          # Antivector-
# The hole's spin per unit mass: intrinsic, the patch of the spatial plane it turns in.
Spin = Bivector.derive.within(Algebra("x+y+z+")).derive.dual  # Antibivector-
Rotor = ga.gatype.rotor()                                    # Rotor+
# The camera's slices x and y, which read screen directions: extrinsic vectors.
ScreenSlice = Vector.derive.within(Algebra("x+y+"))          # Vector+
# Directions on the camera's screen, along its own x and y: intrinsic antivectors.
Screen = ScreenSlice.derive.dual                             # Antivector-
ScreenFrame = ga.gatype((Antivector, Screen))                # Antivector- <- Screen-
# The light's polarization covariance, from the slices that read the screen to screen directions.
Coherency = ga.gatype((Screen, ScreenSlice))                 # Screen- <- ScreenSlice+
# The null direction's derivative and the position gauge field map directions to directions.
DirectionGradient = ga.gatype((Antivector, Antivector))      # Antivector- <- Antivector-
GaugeField = ga.gatype((Antivector, Antivector))             # Antivector- <- Antivector-
# One unit along the lab's time, and the spatial projection across the lab's slice of space.
time = -mv.t.dual()                                          # [] Antivector-
space = Antivector - time * (mv.t & Antivector)              # [] Antivector- <- Antivector-
# Unit directions along time and space, and the camera's rest-frame axes looking along -z.
axes = stack([time, mv.x.dual(), mv.y.dual(), mv.z.dual()])  # [4] Antivector-
camera_forward = -mv.z.dual()                                # [] Antivector-
camera_right = mv.y.dual()                                   # [] Antivector-
camera_up = -mv.x.dual()                                     # [] Antivector-
# Gravitational constant, speed of light, and reference black-hole mass equal one.
mass = 1.0

## Kerr–Schild geometry

At each spacetime event, ingoing Cartesian Kerr–Schild coordinates specify a scalar profile falling off with oblate radius and an ingoing null direction whose tangent plane `-null.dual()` touches the light cone along it. Adding the profile-weighted null dyad to the identity forms the position gauge field `gauge(profile, null)`, and because the null direction lies in its own tangent plane, the dyad squares to zero and `gauge_inverse(profile, null)` subtracts the same dyad directly. Carrying a timelike direction back into the local frame gives its squared proper interval as minus its flat square. The hole's spin is modelled as an intrinsic spatial patch whose dual inverse `spin.dual_inverse()` is the turning plane that drags outgoing directions, while pairing an orbital plane with the spin `orbit & spin` measures prograde or retrograde alignment.

In [ ]:
# Three sample events at coordinates (t, x, y, z) around a black hole with tilted spin.
event_coordinates = np.array([[0, 4, 2, 1], [3, -3, 1, 2], [-1, 2, -4, -1]])
events = (axes * event_coordinates).sum(axis=-1)                                # [3] Antivector-
tilted_spin = (0.7 * mv.xy + 0.2 * mv.yz).dual()                                # [] Spin-

In [ ]:
def field_components(position: Antivector, spin: Spin, mass: float) -> tuple[Scalar, Scalar, Antivector, Vector, DirectionGradient, Bivector]:
    """Kerr–Schild geometry at `position` comprises the oblate radius, scalar profile, ingoing null direction, their gradients, and the null curl (`[...]`)."""
    spatial = space(position)                                                               # [...] Antivector-
    # The plane the hole turns in, whose dual is its spin.
    turning = spin.dual_inverse()                                                           # [...] Bivector+
    # The volume the position spans with the turning plane: the plane times the position's height off it,
    # held by its complement along time. Wedged with the plane it is that height along the spin, times the
    # spin squared; the plane enters twice, so its sense of turning drops out.
    volume = spatial.anticommutator(turning)                                                # [...] Vector+
    height = volume ^ turning                                                               # [...] Antivector-
    spin_squared = spin.squared()                                                           # [...] Scalar+
    offset = spatial.squared() - spin_squared                                               # [...] Scalar+
    discriminant = (offset.squared() - 4 * volume.squared()).square_root()                  # [...] Scalar+
    radius_squared = (offset + discriminant) / 2                                            # [...] Scalar+
    radius = radius_squared.square_root()                                                   # [...] Scalar+
    # Paired with a small step, how far the step changes the oblate radius.
    radius_gradient = (radius_squared * spatial - height).dual() / (radius * discriminant)  # [...] Vector+
    # The hole turns as `(turning * (-angle / 2)).exp()` does, and drags the frame with it: the outgoing
    # direction twists against the turning, by the position's commutator with the plane.
    direction = (radius * spatial - spatial.commutator(turning) - height / radius) / (radius_squared + spin_squared)  # [...] Antivector-
    lever = spatial - 2 * radius * direction + height / radius_squared                      # [...] Antivector-
    direction_gradient = (
        lever * (radius_gradient & Antivector)
        + radius * space - space.commutator(turning)
        - (space.anticommutator(turning) ^ turning) / radius
    ) / (radius_squared + spin_squared)                                                     # [...] Antivector- <- Antivector-
    denominator = radius_squared.squared() - volume.squared()                               # [...] Scalar+
    profile = mass * radius**3 / denominator                                                # [...] Scalar+
    profile_gradient = profile * (3 * radius_gradient / radius
        - (4 * radius**3 * radius_gradient - 2 * height.dual()) / denominator)              # [...] Vector+
    # The curl of the null vector, `-(Vector ^ null_gradient(Antivector).dual()).trace(1, 2)`, piece by
    # piece: the rank-one piece gives its two vectors wedged, the twist by the plane twice the plane, the
    # rest none.
    null_curl = ((radius_gradient ^ lever.dual()) - 2 * turning) / (radius_squared + spin_squared)  # [...] Bivector+
    return radius, profile, time - direction, profile_gradient, -direction_gradient, null_curl


def dyad(profile: Scalar, null: Antivector) -> GaugeField:
    """The profile-weighted null dyad (`[...] Antivector- <- Antivector-`) pairs directions with the null tangent plane `-null.dual()`."""
    null_plane = -null.dual()                                                               # [...] Vector+
    return profile * null * (null_plane & Antivector)                                       # [...] Antivector- <- Antivector-

In [ ]:
def gauge(profile: Scalar, null: Antivector) -> GaugeField:
    """The position gauge field (`[...] Antivector- <- Antivector-`) maps local-frame directions to Kerr–Schild coordinate directions."""
    return Antivector + dyad(profile, null)                                                 # [...] Antivector- <- Antivector-


def gauge_inverse(profile: Scalar, null: Antivector) -> GaugeField:
    """The inverse position gauge field (`[...] Antivector- <- Antivector-`) maps Kerr–Schild coordinate directions back to the local frame."""
    return Antivector - dyad(profile, null)                                                 # [...] Antivector- <- Antivector-

In [ ]:
def proper_squared(profile: Scalar, null: Antivector, direction: Antivector) -> Scalar:
    """Squared proper length (`[...] Scalar+`) of a timelike direction equals minus its flat square in the local frame."""
    return -gauge_inverse(profile, null)(direction).squared()                               # [...] Scalar+


event_radius, event_profile, event_null, _, _, _ = field_components(events, tilted_spin, mass)
clock_squared = proper_squared(event_profile, event_null, time)                             # [3] Scalar+
with np.printoptions(precision=3, suppress=True):
    print("oblate radius:             ", event_radius.to_array())
    print("scalar profile:            ", event_profile.to_array())
    print("static clock rate squared: ", clock_squared.to_array())

## Equatorial rays across black-hole spins

Raising a ray's momentum with the flat metric and carrying it through `gauge(profile, null)` gives its local-frame direction, and minus half that direction's flat square is the ray Hamiltonian; the leading minus sign matches the negative square of `time` in the algebra so timelike momenta have positive square in the physical metric convention. Differentiating the Hamiltonian with respect to momentum applies `gauge(profile, null)` a second time to yield the ray tangent, while pulling the momentum back through `null_gradient.adjugate()(momentum)` combines with the profile gradient to yield minus the momentum rate. Parallel rays launched across an equatorial black hole whose spin sweeps from retrograde to prograde follow fourth-order Runge–Kutta steps that taper smoothly inside the horizon.

In tensor-index notation, the ray Hamiltonian reads $H = \tfrac{1}{2} g^{\mu\nu} p_\mu p_\nu$, with Hamilton's equations $\dot{x}^\mu = \partial H / \partial p_\mu$ and $\dot{p}_\mu = -\partial H / \partial x^\mu$.

In [ ]:
n_frames = 160
n_sweep_frames = n_frames // 2 + 1
max_spin = 0.8 * mass
# One sweep of the spin from retrograde to prograde; the animation plays it forward and back.
spins = -max_spin * np.cos(np.linspace(0, np.pi, n_sweep_frames))
frame_duration_ms = 40
extent = 12.0 * mass
n_rays = 31
offsets = np.linspace(-8, 8, n_rays) * mass
step_size, n_steps = 0.1 * mass, 400
stop_fraction = 0.8
n_ring_samples = 128

In [ ]:
def local_direction(profile: Scalar, null: Antivector, momentum: Vector) -> Antivector:
    """The local-frame direction (`[...] Antivector-`) raises `momentum` flat and carries it through the position gauge field."""
    return gauge(profile, null)(-momentum.dual())                                           # [...] Antivector-


def hamiltonian(profile: Scalar, null: Antivector, momentum: Vector) -> Scalar:
    """The ray Hamiltonian (`[...] Scalar+`) equals minus half the flat square of the momentum's local-frame direction."""
    return -local_direction(profile, null, momentum).squared() / 2                          # [...] Scalar+

In [ ]:
def overlap_gradient(null_gradient: DirectionGradient, momentum: Vector) -> Vector:
    """The adjugate `null_gradient.adjugate()` pulls `momentum` back to the gradient (`[...] Vector+`) of `momentum & null`."""
    return null_gradient.adjugate()(momentum)                                               # [...] Vector+


def motion(profile: Scalar, null: Antivector, profile_gradient: Vector,
           null_gradient: DirectionGradient, momentum: Vector) -> tuple[Antivector, Vector]:
    """Hamiltonian differentiation gives the ray tangent (`[...] Antivector-`) and minus the position gradient (`[...] Vector+`)."""
    overlap = momentum & null                                                               # [...] Scalar+
    # Minus half the flat square of the momentum's local direction is the Hamiltonian; its momentum derivative
    # is that direction carried through the gauge field once more.
    tangent = gauge(profile, null)(local_direction(profile, null, momentum))                # [...] Antivector-
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (profile_gradient * overlap.squared()
                + 2 * profile * overlap * overlap_gradient(null_gradient, momentum))        # [...] Vector+
    return tangent, -gradient                                                               # [...] Antivector-, [...] Vector+

In [ ]:
def rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Hamiltonian ray dynamics at `position` pairs the ray tangent (`[...] Antivector-`) with the momentum rate (`[...] Vector+`)."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, null_curl
    return motion(profile, null, profile_gradient, null_gradient, momentum)                 # [...] Antivector-, [...] Vector+


def initial_momentum(position: Antivector, direction: Antivector, spin: Spin, mass: float) -> Vector:
    """A future-directed null momentum (`[...] Vector+`) starts from unit spatial `direction` in the local Kerr–Schild frame."""
    radius, profile, null, _, _, _ = field_components(position, spin, mass)
    del radius
    local_null = time + direction                                                           # [...] Antivector-
    # Carried back out of the local frame and lowered flat.
    momentum = -gauge_inverse(profile, null)(local_null).dual()                             # [...] Vector+
    return momentum / (momentum & time)                                                     # [...] Vector+

In [ ]:
sweep_spin = mv.xy.dual() * spins[:, None]                                                  # [n_sweep_frames, 1] Spin-
sweep_start = -mv.x.dual() * extent + mv.y.dual() * offsets                                 # [n_rays] Antivector-
sweep_momentum = initial_momentum(sweep_start, mv.x.dual(), sweep_spin, mass)               # [n_sweep_frames, n_rays] Vector+
_, sweep_profile, sweep_null, _, _, _ = field_components(sweep_start, sweep_spin, mass)
sweep_hamiltonian = hamiltonian(sweep_profile, sweep_null, sweep_momentum)                  # [n_sweep_frames, n_rays] Scalar+
sweep_position = sweep_start.broadcast_to(sweep_momentum.shape)                             # [n_sweep_frames, n_rays] Antivector-
paths, _ = trace(sweep_position, sweep_momentum, sweep_spin, mass, step_size, n_steps, stop_fraction, rates)  # [n_steps + 1, n_sweep_frames, n_rays] Antivector-

# A constant oblate radius cuts the equatorial plane in a circle of coordinate radius sqrt(horizon^2 + spin^2).
horizon = mass + np.sqrt(mass**2 - spins**2)                                                # [n_sweep_frames]
horizon_radius = np.sqrt(horizon**2 + spins**2)                                             # [n_sweep_frames]
angles = np.linspace(0, 2 * np.pi, n_ring_samples, endpoint=False)
turns = (mv.xy * (-angles / 2)).exp()                                                       # [n_ring_samples] Rotor+
horizons = (turns >> mv.x.dual()) * horizon_radius[:, None]                                 # [n_sweep_frames, n_ring_samples] Antivector-
show_animation(render.ray_animation(paths, horizons, extent), "kerr_spin", frame_duration_ms, forward_then_back=True);

Ray colours run from cool blue on one side of the incoming fan to warm red on the other, while the dark central disc marks the equatorial section of the event horizon. As spin reverses, frame dragging sweeps the critical impact parameter from one side to the other: rays counter-rotating against the hole require larger impact parameters to escape, whereas co-rotating rays skim closer to the horizon and whirl further around before emerging.

## Lensed accretion disk and camera screen transport

Each pixel's past-directed ray carries the camera's transverse screen back toward the source by parallel transport. In gauge theory gravity, [Francis and Kosowsky (2004)](https://arxiv.org/pdf/gr-qc/0311007) express the spin connection along a local direction as a rotation gauge bivector, the spacetime plane in which the local frame turns or boosts per step; taking `-connection.commutator(screen)` transports both screen axes simultaneously.

In [ ]:
def rotation(profile: Scalar, null: Antivector, profile_gradient: Vector,
             null_gradient: DirectionGradient, null_curl: Bivector, direction: Antivector) -> Bivector:
    """The rotation gauge bivector (`[...] Bivector+`) gives the plane in which the local frame turns or boosts along `direction`."""
    null_plane = -null.dual()                                                               # [...] Vector+
    projection = null_plane & direction                                                     # [...] Scalar+
    slope = profile_gradient * projection + profile * overlap_gradient(null_gradient, -direction.dual())  # [...] Vector+
    return -(slope ^ null_plane) - profile * null_curl * projection                         # [...] Bivector+


def polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Past-directed ray motion carries the camera screen (`[...] Antivector- <- Screen-`) by parallel transport per unit of coordinate time."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius
    tangent, momentum_rate = motion(profile, null, profile_gradient, null_gradient, momentum)
    heading = local_direction(profile, null, momentum)                                      # [...] Antivector-
    connection = rotation(profile, null, profile_gradient, null_gradient, null_curl, heading)  # [...] Bivector+
    screen_rate = -connection.commutator(screen)                                            # [...] Antivector- <- Screen-
    clock = -(mv.t & tangent)                                                               # [...] Scalar+
    return tangent / clock, momentum_rate / clock, screen_rate / clock                      # [...] Antivector-, [...] Vector+, [...] Antivector- <- Screen-

Equatorial gas orbits the hole at Keplerian coordinate angular speed down to the innermost stable circular orbit of Bardeen, Press and Teukolsky (1972) and radiates with the relativistic thin-disk temperature profile of [Page and Thorne (1974)](https://doi.org/10.1086/152990), vanishing at the inner edge, peaking just outside it, and falling off at large radius. Pairing a ray's conserved momentum with the static camera's four-velocity and with the orbiting gas gives the observed-to-emitted frequency ratio `(momentum & at_rest) / (momentum & gas)`, which rescales a blackbody spectrum directly in temperature while slant path length in the gas rest frame sets the fraction absorbed at each disk crossing.

In [ ]:
def orbital_speed(radius: Scalar, orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Keplerian coordinate angular speed (`[...] Scalar+`) of equatorial gas depends on oblate `radius` and spin alignment `orbit & spin`."""
    alignment = orbit & spin                                                                # [] Scalar+
    return np.sqrt(mass) / (radius * radius.square_root() + alignment * np.sqrt(mass))      # [...] Scalar+


def orbiting_gas(position: Antivector, orbit: Bivector, spin: Spin, mass: float) -> Antivector:
    """Equatorial gas on a circular orbit in `orbit` has unit four-velocity (`[...] Antivector-`)."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del profile_gradient, null_gradient, null_curl
    angular_speed = orbital_speed(radius, orbit, spin, mass)                                # [...] Scalar+
    # A point turned by `(orbit * (-angle / 2)).exp()` moves by its commutator with the orbit's plane.
    flow = time + angular_speed * space(position).commutator(orbit)                         # [...] Antivector-
    return flow / proper_squared(profile, null, flow).square_root()                         # [...] Antivector-

In [ ]:
def frequency_ratio(position: Antivector, momentum: Vector, camera: Antivector, orbit: Bivector,
                    spin: Spin, mass: float) -> Scalar:
    """Observed-to-emitted frequency ratio (`[...] Scalar+`) compares momentum pairings between a static camera at `camera` and orbiting gas."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector-
    camera_radius, camera_profile, camera_null, _, _, _ = field_components(camera, spin, mass)
    del camera_radius
    at_rest = time / proper_squared(camera_profile, camera_null, time).square_root()        # [] Antivector-
    # The momentum paired with time is conserved along the ray, so the camera's frequency reads here.
    return (momentum & at_rest) / (momentum & gas)                                          # [...] Scalar+


def cube_root(value: Scalar) -> Scalar:
    """Real cube roots (`[...] Scalar+`) of non-negative scalars follow from logarithmic scaling."""
    return (value.log() / 3).exp()                                                          # [...] Scalar+

In [ ]:
def innermost_stable_orbit(orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """The innermost stable circular equatorial orbit radius (`[] Scalar+`) follows Bardeen, Press and Teukolsky (1972)."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar+
    first = 1 + cube_root(1 - ratio**2) * (cube_root(1 + ratio) + cube_root(1 - ratio))     # [] Scalar+
    second = (3 * ratio**2 + first**2).square_root()                                        # [] Scalar+
    # Gas orbiting with the hole reaches inward of the radius without spin, gas orbiting against it outward.
    reach = ((3 - first) * (3 + first + 2 * second)).square_root()                          # [] Scalar+
    return mass * (3 + second - where(ratio < 0, -reach, reach))                            # [] Scalar+


def disk_temperature(radius: Scalar, orbit: Bivector, spin: Spin, mass: float, scale: float) -> Scalar:
    """The Page and Thorne (1974) thin-disk temperature (`[...] Scalar+`) vanishes at the innermost stable orbit and peaks just outside it."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar+
    root_radius = (radius / mass).square_root()                                             # [...] Scalar+
    edge = (innermost_stable_orbit(orbit, spin, mass) / mass).square_root()                 # [] Scalar+
    # The roots of the cubic `root_radius**3 - 3 * root_radius + 2 * ratio`, and the weight of each in
    # the torque; the product of a root's differences from the other two is the cubic's derivative there.
    roots = 2 * ((ratio.arccos() + np.array([-np.pi, np.pi, 3 * np.pi])) / 3).cos()         # [3] Scalar+
    weights = (roots - ratio) ** 2 / (roots * (roots ** 2 - 1))                             # [3] Scalar+
    # The torque the disk carries, zero at the innermost stable orbit; round-off kept non-negative.
    torque = (root_radius - edge - 1.5 * ratio * (root_radius / edge).log()
              - (weights * ((root_radius[..., None] - roots) / (edge - roots)).log()).sum(axis=-1)).clip(0, np.inf)  # [...] Scalar+
    flux = 3 / (8 * np.pi) * torque / (root_radius ** 4 * (root_radius ** 3 - 3 * root_radius + 2 * ratio))  # [...] Scalar+
    return scale * flux.square_root().square_root()                                         # [...] Scalar+

In [ ]:
def disk_depth(radius: Scalar, inner_radius: Scalar, outer_radius: float, depth: float) -> Scalar:
    """Perpendicular optical depth (`[...] Scalar+`) tapers quadratically from `depth` at `inner_radius` to zero at `outer_radius`."""
    fraction = (radius - inner_radius) / (outer_radius - inner_radius)                      # [...] Scalar+
    return depth * (1 - fraction) * (1 - fraction)                                          # [...] Scalar+


def disk_opacity(position: Antivector, momentum: Vector, depth: Scalar, normal: Vector,
                 orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Slant optical depth in the orbiting gas rest frame sets the absorbed ray fraction (`[...] Scalar+`) across the disk."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector-
    slant = ((momentum & gas) / (momentum & normal.dual())).abs()                           # [...] Scalar+
    return 1 - (-depth * slant).exp()                                                       # [...] Scalar+

A coordinate-static camera outside the static limit boosts its rest-frame sightlines and transverse screen axes into the local Kerr–Schild frame before following every pixel's ray backward through up to four equatorial crossings or out to the background sky. Tracing the camera's rays and transporting their screens takes about thirty seconds.

In [ ]:
camera_spin = 0.8 * mass
camera_distance = 28.0 * mass
camera_inclination = np.deg2rad(75.0)
half_view = np.tan(np.deg2rad(19.0))
n_image_width = 320
n_image_height = n_image_width * 2 // 3
n_pixels = n_image_width * n_image_height
camera_step_fraction, n_camera_steps = 0.2, 720
disk_outer_radius = 12.0 * mass
# The disk's optical depth straight through its thickness, at its inner edge.
inner_depth = 4.0
# Accretion temperature scale in kelvin, and mid-grey reference exposure temperature.
disk_temperature_scale = 58500.0
exposure_temperature = 6500.0
escape_radius = 100.0 * mass
# Captured inside every photon orbit, where no ray turns back out.
capture_radius = 1.05 * (mass + np.sqrt(mass**2 - camera_spin**2))
n_sky_width, sky_seed = 2048, 8
n_sky_height = n_sky_width // 2

In [ ]:
def camera_boost(profile: Scalar, null: Antivector) -> Rotor:
    """The Lorentz boost rotor (`[...] Rotor+`) carries a coordinate-static camera's rest frame into the local Kerr–Schild frame."""
    observer = gauge_inverse(profile, null)(time) / proper_squared(profile, null, time).square_root()  # [...] Antivector-
    return (1 - observer * time).normalized()                                               # [...] Rotor+


def camera_rays(eye: Antivector, orientation: Rotor, n_width: int, n_height: int,
                half_view: float, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Pixel-centre sightlines of a coordinate-static camera yield ray origins (`[n_height * n_width] Antivector-`) and past-directed momenta (`[n_height * n_width] Vector+`)."""
    horizontal = ((np.arange(n_width) + 0.5) / n_width * 2 - 1) * half_view * n_width / n_height
    vertical = (1 - (np.arange(n_height) + 0.5) / n_height * 2) * half_view
    direction = orientation >> (camera_forward + camera_right * horizontal[None, :] + camera_up * vertical[:, None])  # [n_height, n_width] Antivector-
    direction = direction / direction.antinorm()                                            # [n_height, n_width] Antivector-
    radius, profile, null, _, _, _ = field_components(eye, spin, mass)
    del radius
    boost = camera_boost(profile, null)                                                     # [] Rotor+
    # The future photon arrives opposite the direction the camera looks; its rest frame is boosted
    # into the Kerr–Schild frame before its heading is lowered into a coordinate momentum.
    heading = boost >> (time - direction)                                                   # [n_height, n_width] Antivector-
    momentum = -gauge_inverse(profile, null)(heading).dual()                                # [n_height, n_width] Vector+
    momentum = -momentum / (momentum & time)                                                # [n_height, n_width] Vector+
    momentum = momentum.reshape(-1)                                                         # [n_height * n_width] Vector+
    # Every ray starts at the eye, as an event of the whole of spacetime, the type its steps carry.
    return eye.broadcast_to(momentum.shape), momentum

In [ ]:
def camera_screen(position: Antivector, momentum: Vector, orientation: Rotor,
                  spin: Spin, mass: float) -> ScreenFrame:
    """Orthonormal transverse camera axes form a screen map (`[...] Antivector- <- Screen-`) across each ray."""
    radius, profile, null, _, _, _ = field_components(position, spin, mass)
    del radius
    boost = camera_boost(profile, null)                                                     # [...] Rotor+
    unboost = boost.reverse()                                                               # [...] Rotor+
    heading = unboost >> local_direction(profile, null, momentum)                           # [...] Antivector-
    direction = space(heading) / -(mv.t & heading)                                          # [...] Antivector-
    # The turn carrying the camera's forward onto the direction it looks to see the ray: the reflection in
    # the forward, then in the direction halfway between the two.
    forward = orientation >> camera_forward                                                 # [] Antivector-
    halfway = direction + forward                                                           # [...] Antivector-
    turn = halfway * forward / halfway.antinorm()                                           # [...] Rotor+
    frame_rotor = boost * turn * orientation                                                # [...] Rotor+
    # The camera's own right and up, turned with it; a screen direction is read by pairing with the
    # camera's slices x and y.
    return frame_rotor >> (camera_right * (mv.x & Screen) + camera_up * (mv.y & Screen))    # [...] Antivector- <- Screen-


orientation = (mv.zx * (-camera_inclination / 2)).exp()                                     # [] Rotor+
eye = (orientation >> mv.z.dual()) * camera_distance                                        # [] Antivector-
spin = mv.xy.dual() * camera_spin                                                           # [] Spin-
inner_radius = innermost_stable_orbit(mv.xy, spin, mass)                                    # [] Scalar+
position, momentum = camera_rays(eye, orientation, n_image_width, n_image_height, half_view, spin, mass)  # [n_pixels] Antivector-, [n_pixels] Vector+
screen = camera_screen(position, momentum, orientation, spin, mass)                         # [n_pixels] Antivector- <- Screen-


def opacity(event: Antivector, ray_momentum: Vector, radius: Scalar) -> Scalar:
    """Disk crossings absorb a ray fraction (`[...] Scalar+`) set by the radial depth profile and slant angle."""
    depth = disk_depth(radius, inner_radius, disk_outer_radius, inner_depth)                # [...] Scalar+
    return disk_opacity(event, ray_momentum, depth, mv.z, mv.xy, spin, mass)                # [...] Scalar+


image, _ = resolve((position, momentum, screen), spin, mass, camera_step_fraction, n_camera_steps,
                   polarized_rates, mv.z, inner_radius, disk_outer_radius, opacity, escape_radius, capture_radius)
points, momenta, screens = image.disk                                                       # [...] Antivector-, [...] Vector+, [...] Antivector- <- Screen-
shift = frequency_ratio(points, momenta, eye, mv.xy, spin, mass)                            # [...] Scalar+
temperature = shift * disk_temperature(image.disk_radii, mv.xy, spin, mass, disk_temperature_scale)  # [...] Scalar+
sky = render.star_texture(n_sky_width, n_sky_height, sky_seed)
save_figure(render.camera_figure(image, temperature, exposure_temperature, (n_image_height, n_image_width), sky), "kerr_camera");

Gravitational lensing bends emission from the far half of the disk over the top of the black shadow and wraps an inverted secondary image underneath, while background starlight arcs around the Einstein ring. Relativistic beaming and Doppler shift brighten and blue-shift the approaching gas on the left relative to the dimmer, red-shifted gas receding on the right, with colour saturation exaggerated to make the temperature shift clear.

## Polarized disk light through a rotating analyzer

Following the prescribed toroidal-field model of [Gelles et al. (2021)](https://arxiv.org/abs/2105.09440), the disk's magnetic field circles its axis in the local Kerr–Schild frame, and synchrotron-like emission is partially polarized perpendicular to that field across the ray, dropping smoothly to unpolarized emission along the field direction. Projecting the emission axis onto the parallel-transported screen builds a unit-intensity polarization covariance `Coherency` from screen slices to screen directions, combining an isotropic metric term for the unpolarized share with the rank-one polarized dyad. Pairing a rotating linear analyzer slice across that map via `analyzers & coherency(analyzers)` reads the transmitted fraction at doubled disk exposure so unpolarized regions match the static view.

In the matrix notation of polarization optics, the polarization state on the two-dimensional screen is written as a $2 \times 2$ coherency matrix $J$, and a unit linear analyzer along $\mathbf{a}$ transmits power $\mathbf{a}^{\mathsf{T}} J\,\mathbf{a}$.

In [ ]:
n_polarizer_frames = n_frames
polarizer_angles = np.linspace(0, np.pi, n_polarizer_frames, endpoint=False)
polarization_degree = 0.8

In [ ]:
def disk_coherency(position: Antivector, momentum: Vector, screen: ScreenFrame,
                   normal: Vector, spin: Spin, mass: float, polarization_degree: float) -> Coherency:
    """Disk emission forms a unit-intensity polarization covariance (`[...] Screen- <- ScreenSlice+`) on the transported camera screen."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, profile_gradient, null_gradient, null_curl
    heading = local_direction(profile, null, momentum)                                      # [...] Antivector-
    direction = space(heading) / (mv.t & heading)                                           # [...] Antivector-
    # The magnetic direction circles the normal, square to the plane through the normal and the
    # emitter. Emission is polarized perpendicular to it across the ray: in that plane, square to the ray.
    meridian = space(position).dual() ^ normal                                              # [...] Bivector+
    polarization_axis = meridian.commutator(direction) / meridian.norm()                    # [...] Antivector-
    polarization = screen.adjoint()(polarization_axis)                                      # [...] Screen-
    # Polarization vanishes smoothly when looking along the magnetic direction.
    polarized_fraction = polarization_degree * polarization.squared()                       # [...] Scalar+
    # Unpolarized light is round, where the metric enters; the polarized part weights the polarization by
    # each slice's reading of it.
    return ((1 - polarized_fraction) / 2 * ScreenSlice.dual()
            + polarization_degree * polarization * (ScreenSlice & polarization))            # [...] Screen- <- ScreenSlice+


def transmitted_power(coherency: Coherency, analyzers: ScreenSlice) -> Scalar:
    """Each unit analyzer slice transmits power (`[...] Scalar+`) read by pairing across `coherency`."""
    return analyzers & coherency(analyzers)                                                 # [...] Scalar+

In [ ]:
coherency = disk_coherency(points, momenta, screens, mv.z, spin, mass, polarization_degree)  # [...] Screen- <- ScreenSlice+
analyzer_rotors = (mv.xy * (-polarizer_angles / 2)).exp()                                    # [n_polarizer_frames] Rotor+
analyzers = analyzer_rotors >> mv.x                                                          # [n_polarizer_frames] ScreenSlice+
transmission = transmitted_power(coherency, analyzers[:, None])                              # [n_polarizer_frames, ...] Scalar+
show_animation(
    render.polarizer_animation(image, temperature, exposure_temperature, transmission, (n_image_height, n_image_width), sky),
    "kerr_polarizer",
    frame_duration_ms,
);

## Orbiting disk clumps and light-travel delay

Because the stationary disk geometry, redshift, and ray paths are time-independent, one ray trace serves every frame of an orbital movie while an illustrative clumpy emissivity pattern rides with the gas. At each frame, every disk crossing is turned backward along its equatorial orbit at its Keplerian angular speed through the elapsed coordinate time plus the ray's arrival time, which is earlier for longer paths around the far side and around the photon ring. Differential rotation winds clumps into trailing spirals, light-travel delay shears the near and far sides, and cross-fading two adjacent orbital loops closes the cycle without a seam. This cell takes about ten seconds, most of it tone-mapping the frames.

In [ ]:
n_flow_frames = n_frames
flow_time = 50.0 * mass

In [ ]:
def orbital_origins(position: Antivector, radius: Scalar, orbit: Bivector, spin: Spin, mass: float,
                    distance: float, elapsed: np.ndarray) -> Antivector:
    """Equatorial rotors turn disk events (`[...] Antivector-`) back along their orbits through `elapsed` plus light-travel delay."""
    emitted = elapsed[..., None] + (mv.t & position) + distance                             # [...] Scalar+
    angle = orbital_speed(radius, orbit, spin, mass) * emitted                              # [...] Scalar+
    rotors = (orbit * (angle / 2)).exp()                                                    # [...] Rotor+
    return rotors >> position                                                               # [...] Antivector-


def disk_flow_states() -> Iterator[Antivector]:
    """Yields retrodicted disk crossing positions (`[2, ...] Antivector-`) for the current and preceding orbital loops."""
    for elapsed in np.linspace(0, flow_time, n_flow_frames, endpoint=False):
        yield orbital_origins(points, image.disk_radii, mv.xy, spin, mass, camera_distance,
                              np.array([elapsed, elapsed - flow_time]))


flow_states = show_states(
    disk_flow_states(),
    lambda flows: render.disk_animation(
        image, temperature, exposure_temperature, flows, n_flow_frames, (n_image_height, n_image_width), sky
    ),
    name="kerr_disk",
    duration_ms=frame_duration_ms,
);

In [ ]:
# checks
# The Kerr-Schild null direction is flat null, the gauge inverse is exact across axes, and static clocks run forward.
assert (event_null.squared().abs() < 1e-10).all()
gauge_round_trip = gauge(event_profile[:, None], event_null[:, None])(
    gauge_inverse(event_profile[:, None], event_null[:, None])(axes)
) - axes
assert (gauge_round_trip.scalar_norm_squared() < 1e-12).all()
assert (clock_squared > 0).all()

# Equatorial sweep rays start on the null cone with unit coordinate energy.
assert (sweep_hamiltonian.abs() < 1e-10).all()
assert (((sweep_momentum & time) - 1).abs() < 1e-10).all()

# Camera rays stay null across disk crossings to integrator accuracy, with bounded weights and unit-trace coherency.
crossing_tangent, _ = rates(points, momenta, spin, mass)
assert ((momenta & crossing_tangent).abs() < 7e-2).all()
assert ((image.disk_weights > 0) & (image.disk_weights <= 1)).all()
assert ((image.sky_weights > 0) & (image.sky_weights <= 1)).all()
assert (temperature >= 0).all() and (temperature > 0).any()
coherency_trace = (ScreenSlice & coherency(ScreenSlice)).contract()
assert ((coherency_trace - 1).abs() < 1e-10).all()
assert ((transmission >= 0) & (transmission <= 1)).all()
assert ((flow_states[0][0] - flow_states[0][1]).antinorm() > 0).all()